# Data Loading 

In [14]:
import pandas as pd
import re
from datasets import load_dataset
from textwrap import wrap
import json
import os

os.makedirs("prepared", exist_ok=True)


In [15]:
import numpy as np

In [16]:
hf_datasets = {}

hf_datasets["customer_support"] = load_dataset("Tobi-Bueck/customer-support-tickets")
hf_datasets["helpdesk_synth"] = load_dataset("Console-AI/IT-helpdesk-synthetic-tickets")
hf_datasets["it_troubleshooting"] = load_dataset("UmerSajid/IT-Troubleshooting-Dataset")
hf_datasets["techqa"] = load_dataset("nvidia/TechQA-RAG-Eval")
hf_datasets["amazon_qa"] = load_dataset("embedding-data/Amazon-QA")

local_paths = {
    "tech_support_dialogue": "data/Troubleshooting Dialogue/tech_support_dataset.csv",
    "routing_tickets": "data/Routing Engine/all_tickets_processed_improved_v3.csv",
    "customer_support_tickets": "data/Routing Engine/customer_support_tickets.csv",
    "it_support_ticket_data": "data/Routing Engine/IT Support Ticket Data.csv",
}

local_dfs = {name: pd.read_csv(path) for name, path in local_paths.items()}


In [17]:
def clean_text(t):
    if t is None:
        return ""
    t = str(t)

    # Remove emails / PII
    t = re.sub(r'\S+@\S+', '[EMAIL]', t)
    t = re.sub(r'\b\d{10,15}\b', '[PHONE]', t)

    # Replace IPs
    t = re.sub(r'\b\d{1,3}(?:\.\d{1,3}){3}\b', '[IP]', t)

    # Preserve error codes
    t = re.sub(r'(ORA-\d+)', r' \1 ', t)
    t = re.sub(r'(0x[0-9A-Fa-f]+)', r' \1 ', t)
    t = re.sub(r'(HTTP\s?[45]\d{2})', r' \1 ', t)
    t = re.sub(r'(SQLSTATE\[\w+\])', r' \1 ', t)

    # Remove HTML
    t = re.sub(r'<[^>]+>', '', t)

    # Normalize whitespace
    t = re.sub(r'\s+', ' ', t).strip()

    return t


# Preparing Hugging Face Datasets

In [18]:
df_hf_cust = hf_datasets["customer_support"]["train"].to_pandas()
df_hf_cust["text"] = (df_hf_cust["subject"].fillna("") + " " + df_hf_cust["body"].fillna("")).apply(clean_text)
df_hf_cust["answer"] = df_hf_cust["answer"]
df_hf_cust["category"] = df_hf_cust["queue"]
df_hf_cust["priority"] = df_hf_cust["priority"]


In [19]:
df_hf_cust.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,text,category
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,None,None,None,None,Wesentlicher Sicherheitsvorfall Sehr geehrtes ...,Technical Support
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,None,None,None,"Account Disruption Dear Customer Support Team,...",Technical Support
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,None,None,None,None,None,Query About Smart Home System Integration Feat...,Returns and Exchanges
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,None,None,None,Inquiry Regarding Invoice Details Dear Custome...,Billing and Payments
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,None,None,None,None,Question About Marketing Agency Software Compa...,Sales and Pre-Sales


In [20]:
df_hf_helpdesk = hf_datasets["helpdesk_synth"]["train"].to_pandas()
df_hf_helpdesk["text"] = (df_hf_helpdesk["subject"].fillna("") + " " + df_hf_helpdesk["description"].fillna("")).apply(clean_text)
df_hf_helpdesk["answer"] = None
df_hf_helpdesk["category"] = df_hf_helpdesk["category"]
df_hf_helpdesk["priority"] = df_hf_helpdesk["priority"]


In [21]:
df_hf_helpdesk.head()

,id,subject,description,priority,category,createdAt,requesterEmail,text,answer
0,1aiu3lrqi,Hey IT! Our network printer keeps disconnecting.,Hey IT! Our network printer keeps disconnectin...,Medium,Network,2024-10-28T18:36:55.004Z,jane.doe@acme.co,Hey IT! Our network printer keeps disconnectin...,None
1,kz5mjjpox,Re: [Acme IT] Re: Ticket #98765 - Access Issue...,This is a follow-up to your previous request #...,High,Network,2024-10-28T18:36:56.156Z,user123@acme.co,Re: [Acme IT] Re: Ticket #98765 - Access Issue...,None
2,86eza0fwq,Software Conflict Causing App Crashes,Hey team! :wave: We're experiencing some inter...,High,Software,2024-10-28T18:36:54.644Z,user@acme.co,Software Conflict Causing App Crashes Hey team...,None
3,jtw509e3n,j.doe@acme.co Google Calendar Setup Assistance,We need to set up a new Google Calendar for te...,Medium,Software,2024-10-28T18:36:54.524Z,j.doe@acme.co,[EMAIL] Google Calendar Setup Assistance We ne...,None
4,tso616mbn,Software Access: Asana Project for Jordan Smith,Jordan Smith is part of the Project Management...,Medium,Software,2024-10-28T18:36:54.816Z,jordan.smith@acme.co,Software Access: Asana Project for Jordan Smit...,None


In [22]:
df_hf_trouble = hf_datasets["it_troubleshooting"]["train"].to_pandas()
df_hf_trouble["text"] = (
    df_hf_trouble["Issue"].fillna("") + " " +
    df_hf_trouble["Symptoms"].fillna("") + " " +
    df_hf_trouble["Common Causes"].fillna("") + " " 
).apply(clean_text)

df_hf_trouble["answer"] = df_hf_trouble["Solution Steps"].apply(clean_text)
df_hf_trouble["category"] = df_hf_trouble["Category"]
df_hf_trouble["priority"] = df_hf_trouble["Severity"]

In [23]:
df_hf_trouble.head()

,ID,Category,Issue,Symptoms,Solution Steps,Severity,Estimated Resolution Time,Common Causes,Keywords,Urdu Solution,Documentation Link,text,answer,category,priority
0,1,Cloud Computing,AWS instance not starting - Variant 30,Instance stuck in 'pending' state,"Check instance status in AWS console, review s...",Low,18 hours,"Typical configuration error, outdated software...","cloud computing, aws, instance stuck in 'pendi...","حل: Check instance status in AWS console, revi...",https://docs.aws.amazon.com/AWSEC2/latest/User...,AWS instance not starting - Variant 30 Instanc...,"Check instance status in AWS console, review s...",Cloud Computing,Low
1,2,Cloud Computing,AWS instance not starting - Variant 24,Instance stuck in 'pending' state,"Check instance status in AWS console, review s...",High,9 hours,"Typical configuration error, outdated software...","cloud computing, aws, instance stuck in 'pendi...","حل: Check instance status in AWS console, revi...",https://docs.aws.amazon.com/AWSEC2/latest/User...,AWS instance not starting - Variant 24 Instanc...,"Check instance status in AWS console, review s...",Cloud Computing,High
2,3,Cloud Computing,AWS instance not starting - Variant 25,Instance stuck in 'pending' state,"Check instance status in AWS console, review s...",Low,16 hours,"Typical configuration error, outdated software...","cloud computing, aws, instance stuck in 'pendi...","حل: Check instance status in AWS console, revi...",https://docs.aws.amazon.com/AWSEC2/latest/User...,AWS instance not starting - Variant 25 Instanc...,"Check instance status in AWS console, review s...",Cloud Computing,Low
3,4,Cloud Computing,AWS instance not starting - Variant 5,Instance stuck in 'pending' state,"Check instance status in AWS console, review s...",High,33 hours,"Typical configuration error, outdated software...","cloud computing, aws, instance stuck in 'pendi...","حل: Check instance status in AWS console, revi...",https://docs.aws.amazon.com/AWSEC2/latest/User...,AWS instance not starting - Variant 5 Instance...,"Check instance status in AWS console, review s...",Cloud Computing,High
4,5,Cloud Computing,AWS instance not starting - Variant 48,Instance stuck in 'pending' state,"Check instance status in AWS console, review s...",Low,7 hours,"Typical configuration error, outdated software...","cloud computing, aws, instance stuck in 'pendi...","حل: Check instance status in AWS console, revi...",https://docs.aws.amazon.com/AWSEC2/latest/User...,AWS instance not starting - Variant 48 Instanc...,"Check instance status in AWS console, review s...",Cloud Computing,Low


In [24]:
df_hf_techqa = hf_datasets["techqa"]["train"].to_pandas()
df_hf_techqa["text"] = df_hf_techqa["question"].apply(clean_text)
df_hf_techqa["answer"] = df_hf_techqa["answer"].apply(clean_text)
df_hf_techqa["category"] = "TechQA"
df_hf_techqa["priority"] = None


In [25]:
df_hf_techqa.head()

,id,question,answer,is_impossible,contexts,text,category,priority
0,TRAIN_Q000,User environment variables no longer getting p...,"To work around the issue, set environment vari...",False,"[{'filename': 'swg21996508.txt', 'text': 'Titl...",User environment variables no longer getting p...,TechQA,None
1,TRAIN_Q001,Netcool/Impact (all versions): How is the Exit...,This is because the Exit() parser function in ...,False,"[{'filename': 'swg21675316.txt', 'text': 'Titl...",Netcool/Impact (all versions): How is the Exit...,TechQA,None
2,TRAIN_Q002,Why are replies going to the DLQ with reason 2...,-,True,[],Why are replies going to the DLQ with reason 2...,TechQA,None
3,TRAIN_Q003,How to configure SSL mutual authentication in ...,The following steps help guide you through the...,False,"[{'filename': 'swg21179559.txt', 'text': 'Titl...",How to configure SSL mutual authentication in ...,TechQA,None
4,TRAIN_Q004,Help with Action required for IIB H.E. V9 & WM...,-,True,[],Help with Action required for IIB H.E. V9 & WM...,TechQA,None


In [26]:
df_hf_amz = hf_datasets["amazon_qa"]["train"].to_pandas()

# Remove any pre-existing standardized columns
for col in ["text", "answer", "category", "priority"]:
    if col in df_hf_amz.columns:
        df_hf_amz = df_hf_amz.drop(columns=[col])

# Build text
df_hf_amz["text"] = df_hf_amz["query"].apply(clean_text)

# Build answer from pos list (REAL ANSWER)
df_hf_amz["answer"] = df_hf_amz["pos"].apply(
    lambda x: x[0].strip() if isinstance(x, np.ndarray) and x.size > 0 else ""
)

df_hf_amz["category"] = "ProductQA"
df_hf_amz["priority"] = None


In [27]:
df_hf_amz.head()

,query,pos,text,answer,category,priority
0,does this fit the z2x version? Thx,[I am not 100% sure. It appears that it does b...,does this fit the z2x version? Thx,I am not 100% sure. It appears that it does ba...,ProductQA,None
1,What material are these dumbbells made out of?,[They say cast iron.. I don't know the exact c...,What material are these dumbbells made out of?,They say cast iron.. I don't know the exact ch...,ProductQA,None
2,What is the length and width of this scooter i...,[below are all the products specs- We are auth...,What is the length and width of this scooter i...,below are all the products specs- We are autho...,ProductQA,None
3,how much bigger is the medium versus the baby ...,[These are no longer at my house as they went ...,how much bigger is the medium versus the baby ...,These are no longer at my house as they went h...,ProductQA,None
4,Does it work with t-mobile,[Yes but it almost caught on fire charging in ...,Does it work with t-mobile,Yes but it almost caught on fire charging in m...,ProductQA,None


# Preparing CSV Datasets 

In [28]:
df_local_trouble = local_dfs["tech_support_dialogue"]
df_local_trouble["text"] = df_local_trouble["Customer_Issue"].apply(clean_text)
df_local_trouble["answer"] = df_local_trouble["Tech_Response"].apply(clean_text)
df_local_trouble["category"] = df_local_trouble["Issue_Category"]
df_local_trouble["priority"] = df_local_trouble["Issue_Status"]


In [29]:
df_local_trouble.head()

,Conversation_ID,Customer_Issue,Tech_Response,Resolution_Time,Issue_Category,Issue_Status,text,answer,category,priority
0,CONV-0001,Cannot connect to Wi-Fi,Clear cache and remove unnecessary programs.,92 minutes,Software,Pending,Cannot connect to Wi-Fi,Clear cache and remove unnecessary programs.,Software,Pending
1,CONV-0002,Software installation failure,Reinstall the printer drivers.,76 minutes,Account,Pending,Software installation failure,Reinstall the printer drivers.,Account,Pending
2,CONV-0003,Cannot connect to Wi-Fi,Clear cache and remove unnecessary programs.,50 minutes,Network,Resolved,Cannot connect to Wi-Fi,Clear cache and remove unnecessary programs.,Network,Resolved
3,CONV-0004,Forgot password,Reset your password using the link provided.,97 minutes,Performance,Pending,Forgot password,Reset your password using the link provided.,Performance,Pending
4,CONV-0005,Software installation failure,Follow the software installation guide.,110 minutes,Performance,Pending,Software installation failure,Follow the software installation guide.,Performance,Pending


In [30]:
df_local_routing = local_dfs["routing_tickets"]
df_local_routing["text"] = df_local_routing["Document"].apply(clean_text)
df_local_routing["answer"] = None
df_local_routing["category"] = df_local_routing["Topic_group"]
df_local_routing["priority"] = None


In [31]:
df_local_routing.head()

,Document,Topic_group,text,answer,category,priority
0,connection with icon icon dear please setup ic...,Hardware,connection with icon icon dear please setup ic...,None,Hardware,None
1,work experience user work experience user hi w...,Access,work experience user work experience user hi w...,None,Access,None
2,requesting for meeting requesting meeting hi p...,Hardware,requesting for meeting requesting meeting hi p...,None,Hardware,None
3,reset passwords for external accounts re expir...,Access,reset passwords for external accounts re expir...,None,Access,None
4,mail verification warning hi has got attached ...,Miscellaneous,mail verification warning hi has got attached ...,None,Miscellaneous,None


In [32]:
df_local_cust = local_dfs["customer_support_tickets"]
df_local_cust["text"] = df_local_cust["Ticket Description"].apply(clean_text)
df_local_cust["answer"] = None
df_local_cust["category"] = df_local_cust.get("Ticket Type")
df_local_cust["priority"] = df_local_cust.get("Ticket Priority")


In [33]:
df_local_cust.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,...,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating,text,answer,category,priority
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,...,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN,I'm having an issue with the {product_purchase...,None,Technical issue,Critical
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,...,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN,I'm having an issue with the {product_purchase...,None,Technical issue,Critical
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,...,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0,I'm facing a problem with my {product_purchase...,None,Technical issue,Low
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,...,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0,I'm having an issue with the {product_purchase...,None,Billing inquiry,Low
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,...,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0,I'm having an issue with the {product_purchase...,None,Billing inquiry,Low


In [34]:
df_local_dept = local_dfs["it_support_ticket_data"]
df_local_dept["text"] = df_local_dept["Body"].apply(clean_text)
df_local_dept["answer"] = None
df_local_dept["category"] = df_local_dept["Department"]
df_local_dept["priority"] = df_local_dept["Priority"]


In [35]:
df_local_dept.head()

,Unnamed: 0,Body,Department,Priority,Tags,text,answer,category,priority
0,0,"Dear Customer Support Team,I am writing to rep...",Technical Support,high,"['Account', 'Disruption', 'Outage', 'IT', 'Tec...","Dear Customer Support Team,I am writing to rep...",None,Technical Support,high
1,1,"Dear Customer Support Team,I hope this message...",Returns and Exchanges,medium,"['Product', 'Feature', 'Tech Support']","Dear Customer Support Team,I hope this message...",None,Returns and Exchanges,medium
2,2,"Dear Customer Support Team,I hope this message...",Billing and Payments,low,"['Billing', 'Payment', 'Account', 'Documentati...","Dear Customer Support Team,I hope this message...",None,Billing and Payments,low
3,3,"Dear Support Team,I hope this message reaches ...",Sales and Pre-Sales,medium,"['Product', 'Feature', 'Feedback', 'Tech Suppo...","Dear Support Team,I hope this message reaches ...",None,Sales and Pre-Sales,medium
4,4,"Dear Customer Support,I hope this message reac...",Technical Support,high,"['Feature', 'Product', 'Documentation', 'Feedb...","Dear Customer Support,I hope this message reac...",None,Technical Support,high


In [36]:
df_all = pd.concat([
    df_hf_cust,
    df_hf_helpdesk,
    df_hf_trouble,
    df_hf_techqa,
    df_hf_amz,
    df_local_trouble,
    df_local_routing,
    df_local_cust,
    df_local_dept
], ignore_index=True)

df_all = df_all[["text", "answer", "category", "priority"]]
df_all = df_all.dropna(subset=["text"])


In [37]:
with open("prepared/lm_corpus.txt", "w", encoding="utf8") as f:
    for _, row in df_all.iterrows():
        f.write(row["text"] + "\n")
        if isinstance(row["answer"], str) and len(row["answer"]) > 1:
            f.write(row["answer"] + "\n")


In [38]:
df_class = df_all[df_all["category"].notna()]
df_class.to_csv("prepared/classification_dataset.csv", index=False)


In [39]:
import json
import random
import pandas as pd

support_instructions = [
    "Provide a technical support response.",
    "Help the user diagnose the issue.",
    "Give step-by-step troubleshooting instructions.",
    "Respond as an IT support assistant.",
    "Explain how to resolve the following technical problem.",
    "Give a concise answer to the IT query.",
]

classification_instructions = [
    "Categorize the IT support ticket.",
    "Identify the correct assignment group.",
    "Predict the correct issue category.",
    "Determine the ticket’s department.",
    "Classify the ticket into a support domain.",
]

priority_instructions = [
    "Predict the urgency of the issue.",
    "Assign a priority level.",
    "Determine if the ticket is high, medium, or low priority.",
]

instr_rows = []

for _, row in df_all.iterrows():
    text = row.get("text", "")
    answer = row.get("answer", None)
    category = row.get("category", None)
    priority = row.get("priority", None)

    if isinstance(answer, str) and len(answer.strip()) > 1:
        instr_rows.append({
            "instruction": random.choice(support_instructions),
            "input": text,
            "output": answer
        })

    if pd.notna(category):
        instr_rows.append({
            "instruction": random.choice(classification_instructions),
            "input": text,
            "output": str(category)
        })

    if pd.notna(priority):
        instr_rows.append({
            "instruction": random.choice(priority_instructions),
            "input": text,
            "output": str(priority)
        })

with open("prepared/instruction_dataset.jsonl", "w", encoding="utf-8") as f:
    for r in instr_rows:
        f.write(json.dumps(r) + "\n")


In [40]:
from textwrap import wrap
import pandas as pd

kb_rows = []

for _, row in df_all.iterrows():

    answer = row.get("answer", None)

    if not isinstance(answer, str) or len(answer.strip()) < 100:
        continue

    for chunk in wrap(answer.strip(), width=800):
        kb_rows.append({
            "text": chunk,
            "source": str(row.get("category", "Unknown"))
        })

df_kb = pd.DataFrame(kb_rows)
df_kb.to_csv("prepared/kb_chunks.csv", index=False)

